# Автосрез agent-live на Kaggle (T4 ×2)Read-only прогон 7 задач комплекта на бесплатном GPU.Перед запуском: **Settings → Accelerator → GPU T4 ×2**, **Settings → Internet → On**.Ячейки выполнять сверху вниз. Модель и репозитории — в первой ячейке.**Чего ноутбук не делает:** не проверяет E-04/E-05/E-06 (правка файла, git diff)и не заменяет ручной прогон AGT-02 в Cline. Это запасной стенд, а не канон.Подробности и границы — `docs/AGENTS-kaggle-runbook.md`.

In [ ]:
KIT_REPO = "https://github.com/elips0675-web/3060OllamaClineOpenCode"PROJECT_REPO = "https://github.com/elips0675-web/ticketscursor"MODEL = "qwen2.5-coder-16k:latest"BASE_MODEL = "qwen2.5-coder:14b"WORK = "/tmp/agent-live"

In [ ]:
import osos.environ.update({    "KIT_REPO": KIT_REPO,    "PROJECT_REPO": PROJECT_REPO,    "MODEL": MODEL,    "BASE_MODEL": BASE_MODEL,    "WORK": WORK,    "OLLAMA_HOST": "127.0.0.1:11434",    "OLLAMA_MODELS": WORK + "/models",})print("рабочий каталог:", WORK)

In [ ]:
!nvidia-smi || echo 'НЕТ GPU: Settings → Accelerator → GPU T4 x2, затем Runtime → Restart'

In [ ]:
%%bashset -euo pipefailmkdir -p "$WORK"if [ ! -x "$WORK/ollama/bin/ollama" ]; then  curl -fsSL https://ollama.com/download/ollama-linux-amd64.tgz -o "$WORK/ollama.tgz"  mkdir -p "$WORK/ollama"  tar -xzf "$WORK/ollama.tgz" -C "$WORK/ollama"fi"$WORK/ollama/bin/ollama" --version

In [ ]:
%%bashset -euo pipefailif [ -d "$WORK/ollama/lib/ollama" ]; then export OLLAMA_LIBRARY_PATH="$WORK/ollama/lib/ollama"; fimkdir -p "$OLLAMA_MODELS"if ! curl -fsS http://127.0.0.1:11434/api/version > /dev/null 2>&1; then  nohup "$WORK/ollama/bin/ollama" serve > "$WORK/ollama-serve.log" 2>&1 &  for i in $(seq 1 30); do    if curl -fsS http://127.0.0.1:11434/api/version; then break; fi    sleep 2  doneficurl -fsS http://127.0.0.1:11434/api/version

In [ ]:
%%bashset -euo pipefail"$WORK/ollama/bin/ollama" pull "$BASE_MODEL"cat > "$WORK/Modelfile" <<EOFFROM $BASE_MODELPARAMETER num_ctx 16384PARAMETER temperature 0.2PARAMETER top_p 0.9EOF"$WORK/ollama/bin/ollama" create "$MODEL" -f "$WORK/Modelfile""$WORK/ollama/bin/ollama" show "$MODEL" | head -20

In [ ]:
%%bashset -euo pipefailexport PATH="$WORK/node/bin:$PATH"if ! command -v node > /dev/null 2>&1; then  curl -fsSL https://nodejs.org/dist/v20.11.1/node-v20.11.1-linux-x64.tar.xz -o "$WORK/node.tar.xz"  mkdir -p "$WORK/node"  tar -xJf "$WORK/node.tar.xz" -C "$WORK/node" --strip-components=1finode -v

In [ ]:
%%bashset -euo pipefailif [ ! -d "$WORK/kit/.git" ]; then git clone --depth 1 "$KIT_REPO" "$WORK/kit"; fiif [ ! -d "$WORK/project/.git" ]; then git clone --depth 1 "$PROJECT_REPO" "$WORK/project"; figit -C "$WORK/kit" log -1 --onelinegit -C "$WORK/project" log -1 --oneline

Если репозиторий закрыт, `git clone` попросит пароль и ячейка упадёт.Тогда добавьте токен в Kaggle Secrets и подставьте его в URL в первой ячейке(`https://<token>@github.com/...`). Токен в текст ноутбука не вписывайте.

In [ ]:
%%bashset -uo pipefailexport PATH="$WORK/node/bin:$PATH"export OLLAMA_HOST=127.0.0.1:11434cd "$WORK/kit"OUT="evals/live-kaggle-$(date +%Y-%m-%d-%H%M).json"STATUS=0node scripts/agent-live.mjs --root "$WORK/project" --host http://127.0.0.1:11434 --model "$MODEL" --out "$OUT" || STATUS=$?echo "exit=$STATUS"echo "OUT=$WORK/kit/$OUT"

In [ ]:
import glob, json, os, shutilfiles = sorted(glob.glob(os.path.join(os.environ["WORK"], "kit", "evals", "live-kaggle-*.json")))assert files, "файла прогона нет: смотрите вывод предыдущей ячейки"path = files[-1]data = json.load(open(path, encoding="utf-8"))rows = data["rows"]passed = sum(1 for r in rows if r["pass"])steps = sum(r["steps"] for r in rows) / len(rows)fabricated = sum(r["fabricated"] for r in rows)print(os.path.basename(path), "—", data.get("model"), "—", data.get("root"))print(f"pass {passed}/{len(rows)}, среднее шагов {steps:.1f}, выдумано {fabricated}")for r in rows:    print(f"{r['id']} {'PASS' if r['pass'] else 'FAIL'} шаги={r['steps']} цитат={r['citations']} — {r['reason']}")shutil.copy(path, os.path.join("/kaggle/working", os.path.basename(path)))print("скопировано в /kaggle/working")

## Что делать с результатом1. Скачать JSON из `/kaggle/working` (правая панель → Output).2. Положить в `evals/` с пометкой, что число снято не на 3060.3. Сравнивать с `evals/live-2026-10-31-full2.json` (7/7, среднее 4.1) только   при той же модели и тех же `tasks.json`.4. Помнить: 7 задач из 10. E-04/E-05/E-06 и AGT-02 этот прогон не закрывает.